# Calibración de distancias mínimas y máximas

Este módulo toma el wireframe tridimensional capturado sobre la mano humana mediante MediaPipe y calcula las distancias **mínimas** y **máximas** correspondientes a cada Grado de Libertad (**DOF**).

Estos valores **empíricos** son estrictamente necesarios para el proceso matemático de **normalización**. La normalización permite ir del movimiento físico humano continuo a un vector proporcional del $0$ (cerrado) al $1000$ (abierto) que exige el protocolo **Modbus TCP** para asignar los valores angulares en la mano robótica RH56DFTP-2R.

**Flujo**
1. **Inicialización**: Cargar las dependencias y encender el flujo de video de la cámara Intel RealSense (lente color) junto con la red neuronal de MediaPipe Hands para la detección de *landmarks* espaciales.
2. **Captura Interactiva**: Ejecutar la celda de calibración y seguir las instrucciones en pantalla, presionando la tecla `c` en 3 fases secuenciales:
   - *Fase 1*: Mano completamente abierta (Captura el vector $D_{max}$ para los 6 motores).
   - *Fase 2*: Puño cerrado normal (Captura el vector $D_{min}$ exclusivamente para los dedos principales 0 al 4).
   - *Fase 3*: Pulgar rotado hacia la palma (Captura el vector $D_{min}$ exclusivamente para el motor 5, evitando colisiones mecánicas).
3. **Exportación de Datos**: Copiar los vectores de calibración resultantes (`D_max` y `D_min`) que el script imprime en la consola, para posteriormente usarlos en el código que teleopera la mano robótica.

**Notas**
- MediaPipe asume imagen tipo espejo (selfie). Como la RealSense no refleja la imagen, la etiqueta `Right` puede corresponder a la mano **izquierda** real. Como sucede esto, se cambia el filtro a `"Left"`. 
- El código divide la captura del estado mínimo en dos partes (`MÍNIMA_DEDOS` y `MÍNIMA_PULGAR`). Esto es crucial: al cerrar un puño normal, el pulgar choca físicamente contra el índice, impidiendo que el motor 5 calcule su rango máximo de rotación hacia adentro.
- Para asegurar que las distancias medidas visualmente coincidan con las extremidades de la mano robótica, el bloque de dibujo omite el esqueleto genérico de 21 puntos. Utiliza un diccionario de pixeles para graficar exactamente los 11 nodos de interés, trazando líneas de colores específicos (verde, amarillo y morado) sobre las uniones exactas que conforman las variables `d_indice`, `d_pulgar_flex`, etc.
---

# Instrucciones del proceso de calibración

1. Coloca tu **mano derecha** frente a la cámara **completamente abierta**.
2. Presiona la tecla `c` para capturar las distancias **máximas**.
3. Luego, cierra la mano en **puño totalmente cerrado** (con el pulgar únicamente **flexionado** al máximo contra el **dedo índice**).
4. Presiona la tecla `c`nuevamente para capturar las distancias **mínimas**.
5. Ahora, abre nuevamente la mano y lleva **solamente** el pulgar hacia la palma.
6. Presiona la tecla `c`otra vesz para capturar la distancia **mínima** correspondiente a la **rotación del pulgar**
7. Presiona `q`para salir en cualquier momento.

# 1. Imports y configuración de MediaPipe + RealSense

In [1]:
import pyrealsense2 as rs
import numpy as np
import cv2
import mediapipe as mp

# Inicializar MediaPipe + Arreglos
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(max_num_hands=1, min_detection_confidence=0.7)
D_max = np.zeros(6)
D_min = np.zeros(6)

# Inicializar Intel RealSense
pipeline = rs.pipeline()
config = rs.config()
config.enable_stream(rs.stream.color, 640, 480, rs.format.bgr8, 30)
pipeline.start(config)

INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1792525085.278397   18120 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1792525085.300100   18119 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


# 2. Ejecución de la calibración

In [2]:
# Variables de estado para la calibración en 3 fases
modo = "MÁXIMA"

print("=== CALIBRACIÓN DE 3 FASES (EVITANDO INTERFERENCIA DEL PULGAR) ===")
print("1. Coloca tu mano COMPLETAMENTE ABIERTA y presiona 'c'.")

try:
    while True:
        frames = pipeline.wait_for_frames()
        color_frame = frames.get_color_frame()
        if not color_frame: continue
        
        color_image = np.asanyarray(color_frame.get_data())
        image_rgb = cv2.cvtColor(color_image, cv2.COLOR_BGR2RGB)
        results = hands.process(image_rgb)
        
        D_actual = np.zeros(6)
        mano_detectada = False

        if results.multi_hand_landmarks and results.multi_handedness:
            for hand_landmarks, handedness in zip(results.multi_hand_landmarks, results.multi_handedness):
                if handedness.classification[0].label == "Left":
                    mano_detectada = True
                    puntos = np.array([[lm.x, lm.y, lm.z] for lm in hand_landmarks.landmark])
                    
                    # Cálculo de distancias (Modelo de 11 nodos)
                    d_indice  = np.linalg.norm(puntos[8] - puntos[5])
                    d_corazon = np.linalg.norm(puntos[12] - puntos[9])
                    d_anular  = np.linalg.norm(puntos[16] - puntos[13])
                    d_menique = np.linalg.norm(puntos[20] - puntos[17])
                    d_pulgar_flex = np.linalg.norm(puntos[4] - puntos[5])
                    d_pulgar_rot  = np.linalg.norm(puntos[4] - puntos[17])
                    
                    D_actual = np.array([d_menique, d_anular, d_corazon, d_indice, d_pulgar_flex, d_pulgar_rot])

                    # --- DIBUJO DEL WIREFRAME (11 Nodos) ---
                    h, w, _ = color_image.shape
                    nodos_interes = [0, 2, 4, 5, 8, 9, 12, 13, 16, 17, 20]
                    pixeles = {idx: (int(hand_landmarks.landmark[idx].x * w), int(hand_landmarks.landmark[idx].y * h)) for idx in nodos_interes}
                    
                    for cx, cy in pixeles.values():
                        cv2.circle(color_image, (cx, cy), 6, (255, 0, 0), cv2.FILLED)

                    cv2.line(color_image, pixeles[0], pixeles[5], (200, 200, 200), 2)
                    cv2.line(color_image, pixeles[0], pixeles[9], (200, 200, 200), 2)
                    cv2.line(color_image, pixeles[0], pixeles[13], (200, 200, 200), 2)
                    cv2.line(color_image, pixeles[0], pixeles[17], (200, 200, 200), 2)
                    cv2.line(color_image, pixeles[0], pixeles[2], (200, 200, 200), 2)
                    cv2.line(color_image, pixeles[2], pixeles[4], (200, 200, 200), 2)

                    cv2.line(color_image, pixeles[5], pixeles[8], (0, 255, 0), 3)
                    cv2.line(color_image, pixeles[9], pixeles[12], (0, 255, 0), 3)
                    cv2.line(color_image, pixeles[13], pixeles[16], (0, 255, 0), 3)
                    cv2.line(color_image, pixeles[17], pixeles[20], (0, 255, 0), 3)
                    cv2.line(color_image, pixeles[5], pixeles[4], (0, 255, 255), 3)
                    cv2.line(color_image, pixeles[17], pixeles[4], (255, 0, 255), 3)

                    # Instrucciones en pantalla según la fase
                    if modo == "MÁXIMA":
                        cv2.putText(color_image, "1. Mano ABIERTA - Presiona 'c'", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
                    elif modo == "MÍNIMA_DEDOS":
                        cv2.putText(color_image, "2. PUNO CERRADO - Presiona 'c'", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
                    elif modo == "MÍNIMA_PULGAR":
                        cv2.putText(color_image, "3. ESTIRA INDICE Y METE PULGAR - Presiona 'c'", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 255), 2)
                    elif modo == "LISTO":
                        cv2.putText(color_image, "Calibracion Completa. Presiona 'q'", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)

        cv2.imshow('Calibracion de Distancias', color_image)

        # --- LÓGICA DE CAPTURA EN 3 FASES ---
        key = cv2.waitKey(1) & 0xFF
        if key == ord('c') and mano_detectada:
            if modo == "MÁXIMA":
                D_max = D_actual.copy()
                print(f"✓ Distancias MÁXIMAS capturadas:\n{D_max}")
                print("\n-> AHORA CIERRA EL PUÑO NORMAL Y PRESIONA 'c'.")
                modo = "MÍNIMA_DEDOS"
                
            elif modo == "MÍNIMA_DEDOS":
                # Guardamos las distancias mínimas SOLO de los dedos 0 al 4
                D_min[0:5] = D_actual[0:5].copy()
                print(f"✓ Distancias MÍNIMAS (Dedos) capturadas:\n{D_min[0:5]}")
                print("\n-> AHORA ESTIRA LOS DEDOS Y LLEVA EL PULGAR SOLO HACIA LA PALMA (Rotacion) Y PRESIONA 'c'.")
                modo = "MÍNIMA_PULGAR"
                
            elif modo == "MÍNIMA_PULGAR":
                # Guardamos la distancia mínima SOLO del motor 5 (Rotación)
                D_min[5] = D_actual[5]
                print(f"✓ Distancia MÍNIMA (Rotación Pulgar) capturada: {D_min[5]:.4f}")
                
                print("\n=== CALIBRACIÓN EXITOSA ===")
                print("Copia y pega estos arreglos en tu script de gatillo o teleoperación:")
                print(f"D_max = np.array([{D_max[0]:.4f}, {D_max[1]:.4f}, {D_max[2]:.4f}, {D_max[3]:.4f}, {D_max[4]:.4f}, {D_max[5]:.4f}])")
                print(f"D_min = np.array([{D_min[0]:.4f}, {D_min[1]:.4f}, {D_min[2]:.4f}, {D_min[3]:.4f}, {D_min[4]:.4f}, {D_min[5]:.4f}])")
                modo = "LISTO"
                
        elif key == ord('q'):
            break

finally:
    pipeline.stop()
    cv2.destroyAllWindows()
    print("Ventanas cerradas.")

=== CALIBRACIÓN DE 3 FASES (EVITANDO INTERFERENCIA DEL PULGAR) ===
1. Coloca tu mano COMPLETAMENTE ABIERTA y presiona 'c'.


/home/udi/miniforge3/envs/RealSense/lib/python3.10/site-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


✓ Distancias MÁXIMAS capturadas:
[0.28605543 0.36289616 0.38792472 0.35208239 0.247459   0.42058193]

-> AHORA CIERRA EL PUÑO NORMAL Y PRESIONA 'c'.
✓ Distancias MÍNIMAS (Dedos) capturadas:
[0.06588642 0.08122548 0.08289539 0.077423   0.1321548 ]

-> AHORA ESTIRA LOS DEDOS Y LLEVA EL PULGAR SOLO HACIA LA PALMA (Rotacion) Y PRESIONA 'c'.
✓ Distancia MÍNIMA (Rotación Pulgar) capturada: 0.1085

=== CALIBRACIÓN EXITOSA ===
Copia y pega estos arreglos en tu script de gatillo o teleoperación:
D_max = np.array([0.2861, 0.3629, 0.3879, 0.3521, 0.2475, 0.4206])
D_min = np.array([0.0659, 0.0812, 0.0829, 0.0774, 0.1322, 0.1085])
Ventanas cerradas.
